> 데이터 점검

전처리 필요 여부를 판단하기 위해 파일 구조, 결측치, 중복, 키 연결 상태를 확인한다.

In [5]:
from pathlib import Path
import pandas as pd
import os

In [6]:
# 현재 실행 위치에서 시작
project_dir = Path.cwd()

# 프로젝트 루트 폴더를 찾을 때까지 상위 폴더로 이동
while project_dir.name != "ott-churn-prediction":
    if project_dir == project_dir.parent:
        raise FileNotFoundError("ott-churn-prediction 폴더를 찾지 못했습니다.")
    project_dir = project_dir.parent

# split된 데이터들이 모여있는 중심경로
data_dir = project_dir / "_data" / "02_interim" / "260501_promotion_split"

# 이 데이터들을 각각 지정함
files = {
    "promotion_0_membership": data_dir / "promotion_0_membership.csv",
    "promotion_0_movie_master": data_dir / "promotion_0_movie_master.csv",
    "promotion_0_user_mapping": data_dir / "promotion_0_user_mapping.csv",
    "promotion_0_view_history": data_dir / "promotion_0_view_history.csv",
    "promotion_1_membership": data_dir / "promotion_1_membership.csv",
    "promotion_1_movie_master": data_dir / "promotion_1_movie_master.csv",
    "promotion_1_user_mapping": data_dir / "promotion_1_user_mapping.csv",
    "promotion_1_view_history": data_dir / "promotion_1_view_history.csv",
}

data = {name: pd.read_csv(path) for name, path in files.items()}


> 파일 구조 확인

In [ ]:
rows = []

for name, df in data.items():
    rows.append({
        "file": name,
        "row_count": len(df),
        "column_count": len(df.columns),
        "columns": list(df.columns),
    })

pd.DataFrame(rows)

In [ ]:
for name, df in data.items():
    print(name)
    print(df.dtypes)
    print()

> 결측치 확인

In [ ]:
missing_rows = []

for name, df in data.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        if missing_count > 0:
            missing_rows.append({
                "file": name,
                "column": column,
                "missing_count": missing_count,
                "missing_rate": missing_count / len(df),
            })

pd.DataFrame(missing_rows)

> 중복 확인

In [ ]:
duplicate_rows = []

check_columns = {
    "membership": ["USER_KEY"],
    "movie_master": ["MOVIE_NUM"],
    "user_mapping": ["USER_KEY", "USER_NUM"],
}

for name, df in data.items():
    for table_name, columns in check_columns.items():
        if table_name in name:
            for column in columns:
                duplicate_rows.append({
                    "file": name,
                    "column": column,
                    "duplicate_count": df[column].duplicated().sum(),
                })

pd.DataFrame(duplicate_rows)

> 키 연결 확인

In [ ]:
key_rows = []

for promotion in [0, 1]:
    membership = data[f"promotion_{promotion}_membership"]
    user_mapping = data[f"promotion_{promotion}_user_mapping"]
    view_history = data[f"promotion_{promotion}_view_history"]
    movie_master = data[f"promotion_{promotion}_movie_master"]

    key_rows.append({
        "promotion": promotion,
        "check": "membership.USER_KEY not in user_mapping.USER_KEY",
        "count": (~membership["USER_KEY"].isin(user_mapping["USER_KEY"])).sum(),
    })

    key_rows.append({
        "promotion": promotion,
        "check": "view_history.USER_NUM not in user_mapping.USER_NUM",
        "count": (~view_history["USER_NUM"].isin(user_mapping["USER_NUM"])).sum(),
    })

    key_rows.append({
        "promotion": promotion,
        "check": "view_history.MOVIE_NUM not in movie_master.MOVIE_NUM",
        "count": (~view_history["MOVIE_NUM"].isin(movie_master["MOVIE_NUM"])).sum(),
    })

pd.DataFrame(key_rows)

> 값 범위 확인

In [ ]:
range_rows = []

numeric_columns = {
    "membership": ["price", "max_screen", "age", "reg_hour"],
    "view_history": ["watch_time(min)", "watch_day", "watch_seq"],
    "movie_master": ["ott_release_month"],
}

for name, df in data.items():
    for table_name, columns in numeric_columns.items():
        if table_name in name:
            for column in columns:
                range_rows.append({
                    "file": name,
                    "column": column,
                    "min": df[column].min(),
                    "max": df[column].max(),
                })

pd.DataFrame(range_rows)